***

Preparing Workspace

***

In [ ]:


## Packages ---

import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import urllib.request, json
pd.set_option('display.max_columns', None)

import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
import plotly.subplots as sp
from plotly.subplots import make_subplots


## Setting file paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data' / 'EPA'
path_main = path_sp / 'Data'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'EPA'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'

path_plots = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring")
print('Export Location: ' + str(path_plots))


## User defined functions ---

path_func = path_config0 / 'Functions.py'
with path_func.open("r") as f:
    exec(f.read())


## Export setting ---

export=False



***

Health_3

***

In [ ]:
indicator_name = 'Health_3'

workbook = f"{indicator_name} MSA EPA.xlsx"
file_in = path_plots / 'Data' / workbook
df_msa = pd.read_excel(file_in, sheet_name = 'MSA')

display(df_msa.head())

In [ ]:

# Set Indicator
indicator_name = 'Health_3'
plot_name = 'aqi'


## Organizing ---


df_plot = df_msa.copy()

df_plot['date_local'] = pd.to_datetime(df_plot['date_local'])
df_plot['Year' ] = df_plot['date_local'].dt.year
df_plot = df_plot[df_plot['Year'] >= 1999]

df_plot = df_plot[df_plot['AQI Daily Maximum'] > 100]


df_plot = pd.DataFrame(df_plot[['MSA', 'Year']].value_counts())
df_plot = df_plot.reset_index()


df_plot = df_plot.sort_values(['MSA', 'Year'], ascending = [True, True])
df_plot = df_plot[df_plot['MSA'] == 'Sacramento--Roseville--Arden-Arcade, CA']
df_plot['moving_avg'] = df_plot['count'].rolling(window=5).mean()
df_plot = df_plot.reset_index(drop=True)


display(df_plot.head())


## Plotting ---


fig = px.bar(df_plot, x='Year', y='count')
fig['data'][0]['marker']['color']='#1F45FC'

fig.add_trace(go.Scatter(x=df_plot["Year"], y=df_plot['moving_avg']
                         , name = '5-Year Average'
                         , mode = 'lines+markers'
                         , line=go.scatter.Line(color="maroon")
                        ))


title = '<b>Days Violating National Ambient Air Quality Standards (Ozone and/or PM2.5)</b>  <br><sup>4-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=20, range=[0, 161])
fig.update_xaxes(tick0=0, dtick=4, range=[1998.5, 2023.5])
fig.update_traces(hovertemplate='Number of days: %{y}<br>%{x}')
# fig.update_layout(showlegend=False)


plot_agol(export=export)